# Campaign Targeting Intelligence
**Data Analyst Portfolio Project - UCI Bank Marketing**

Business question: **If the campaign team can only contact a limited share of customers, who should be contacted first?**

This notebook reproduces the core analysis used in the portfolio presentation.


In [1]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score
from xgboost import XGBClassifier

df = pd.read_csv("../data/bank-additional-full.csv", sep=";")
df["target"] = (df["y"] == "yes").astype(int)
df.shape, df["target"].mean()


((41188, 22), np.float64(0.11265417111780131))

## Train and test split
The source file is ordered by date. The earlier 80% is used for training and the later 20% for testing.

`duration` is excluded because it is not available when the pre-call contact list is created.


In [2]:
split = int(len(df) * 0.8)
train = df.iloc[:split].copy()
test = df.iloc[split:].copy()
feature_cols = [c for c in df.columns if c not in ["y", "target", "duration"]]
X_train, X_test = train[feature_cols], test[feature_cols]
y_train, y_test = train["target"], test["target"]
print(len(train), y_train.mean(), len(test), y_test.mean())


32950 0.0637329286798179 8238 0.3083272638990046


## Model comparison
PR-AUC is the main model-selection metric because the final use is customer ranking under class imbalance.


In [3]:
categorical = X_train.select_dtypes(include="object").columns.tolist()
numeric = [c for c in feature_cols if c not in categorical]
def make_preprocessor():
    return ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), categorical), ("num", StandardScaler(), numeric)])
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
models = {
    "Logistic Regression — baseline": LogisticRegression(max_iter=3000, random_state=42),
    "Logistic Regression — class weighted": LogisticRegression(max_iter=3000, class_weight="balanced", random_state=42),
    "Random Forest — class weighted": RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1),
    "XGBoost — class weighted": XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9, scale_pos_weight=scale_pos_weight, eval_metric="logloss", random_state=42, n_jobs=-1)
}
results=[]; fitted={}
for name, model in models.items():
    pipe=Pipeline([("prep",make_preprocessor()),("model",model)])
    pipe.fit(X_train,y_train)
    prob=pipe.predict_proba(X_test)[:,1]
    pred=(prob>=0.5).astype(int)
    results.append({"Model":name,"Accuracy":accuracy_score(y_test,pred),"Precision":precision_score(y_test,pred,zero_division=0),"Recall":recall_score(y_test,pred,zero_division=0),"F1":f1_score(y_test,pred,zero_division=0),"ROC_AUC":roc_auc_score(y_test,prob),"PR_AUC":average_precision_score(y_test,prob)})
    fitted[name]=(pipe,prob)
results=pd.DataFrame(results).sort_values("PR_AUC",ascending=False)
results


,Model,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC
0,Logistic Regression — baseline,0.724690,0.594444,0.337008,0.430151,0.748321,0.526834
1,Logistic Regression — class weighted,0.447560,0.352241,0.943701,0.513002,0.740902,0.520589
2,Random Forest — class weighted,0.690702,0.166667,0.000787,0.001567,0.716165,0.490061
3,XGBoost — class weighted,0.639233,0.415559,0.418504,0.417026,0.610958,0.412087


## Contact-capacity scenarios
Each row is cumulative from the top of the ranked list. Cumulative lift can move up or down across cutoffs on a finite test set because predicted scores do not perfectly order actual outcomes.


In [4]:
best_name=results.iloc[0]["Model"]
best_pipe,best_prob=fitted[best_name]
order=np.argsort(-best_prob)
total_subscribers=int(y_test.sum())
rows=[]
for pct in [0.05,0.10,0.20,0.30]:
    n=int(np.floor(len(y_test)*pct))
    idx=order[:n]
    subscribers=int(y_test.iloc[idx].sum())
    positive_rate=subscribers/n
    capture_rate=subscribers/total_subscribers
    lift=positive_rate/y_test.mean()
    rows.append([pct,n,subscribers,capture_rate,positive_rate,lift])
pd.DataFrame(rows,columns=["capacity","customers","subscribers_captured","capture_rate","positive_rate","cumulative_lift"])


,capacity,customers,subscribers_captured,capture_rate,positive_rate,cumulative_lift
0,0.05,411,227,0.089370,0.552311,1.791316
1,0.10,823,469,0.184646,0.569866,1.848252
2,0.20,1647,979,0.385433,0.594414,1.927867
3,0.30,2471,1362,0.536220,0.551194,1.787691


## Explainability
The selected model is Logistic Regression, so coefficients are used for explanation. They describe associations in the fitted model, not causal effects. Some macroeconomic variables are highly correlated.


In [5]:
prep=best_pipe.named_steps["prep"]
model=best_pipe.named_steps["model"]
coef=pd.DataFrame({"feature":prep.get_feature_names_out(),"coefficient":model.coef_[0]})
coef["abs_coefficient"]=coef["coefficient"].abs()
coef.sort_values("abs_coefficient",ascending=False).head(15)


,feature,coefficient,abs_coefficient
43,cat__month_oct,1.748248,1.748248
42,cat__month_nov,-1.054089,1.054089
41,cat__month_may,-0.963996,0.963996
56,num__emp.var.rate,-0.958034,0.958034
59,num__euribor3m,0.871657,0.871657
40,cat__month_mar,0.727904,0.727904
49,cat__poutcome_failure,-0.538944,0.538944
39,cat__month_jun,-0.536272,0.536272
34,cat__contact_telephone,-0.484852,0.484852
25,cat__default_unknown,-0.434925,0.434925


In [6]:
macro=["emp.var.rate","cons.price.idx","cons.conf.idx","euribor3m","nr.employed"]
df[macro].corr().round(3)


,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed
emp.var.rate,1.000,0.775,0.196,0.972,0.907
cons.price.idx,0.775,1.000,0.059,0.688,0.522
cons.conf.idx,0.196,0.059,1.000,0.278,0.101
euribor3m,0.972,0.688,0.278,1.000,0.945
nr.employed,0.907,0.522,0.101,0.945,1.000


## Measurement
The model predicts subscription propensity, not causal campaign uplift.

For a future randomized campaign:

**Incremental subscription rate = Target group rate − Holdout group rate**

Campaign cost, customer value and true incremental impact require real campaign results.
